# Obtectives

### English

- Load a trained model artifact.

- Load current team vectors.

- Build a prediction-ready match vector.

- Verify feature compatibility with the trained model.

- Generate the first real World Cup 2026 prediction.

- Estimate class probabilities.

- Decode the predicted match outcome.


### Español

- Cargar un artefacto de modelo previamente entrenado.

- Cargar los vectores actuales de los equipos.

- Construir un vector de partido listo para inferencia.

- Verificar la compatibilidad de las variables con el modelo entrenado.

- Generar la primera predicción real del Mundial 2026.

- Estimar las probabilidades de cada resultado posible.

- Decodificar el resultado predicho.


# Configuration

## Imports

In [1]:
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

## Paths

In [4]:
# Models
MODELS_DIR = Path("../models")

# Processed data
PROCESSED_DATA_DIR = Path("../data/processed")

# Model artifacts
RF_MODEL_PATH = MODELS_DIR / "random_forest_v03.pkl"
XGB_MODEL_PATH = MODELS_DIR / "xgboost_v04.pkl"

# Current team vectors
TEAM_VECTORS_PATH = PROCESSED_DATA_DIR / "current_team_vectors_v01.csv"

## Utility Functions

In [30]:
def get_team_vector(
    team_name,
    team_vectors,
    statistical_columns,
    team_id_column="country_clean",
):
    """
    Retrieve the statistical vector of a national team.

    Parameters
    ----------
    team_name : str
        Name of the national team.

    team_vectors : pandas.DataFrame
        Dataset containing one aggregated vector per team.

    statistical_columns : list of str
        Statistical feature columns to retrieve.

    team_id_column : str, default="country_clean"
        Column used to identify each team.

    Returns
    -------
    pandas.Series
        Statistical vector of the requested team.

    Raises
    ------
    ValueError
        If the team is not available or appears more than once.
    """

    team_rows = team_vectors.loc[
        team_vectors[team_id_column].eq(team_name)
    ]

    if team_rows.empty:
        available_teams = sorted(
            team_vectors[team_id_column].unique()
        )

        raise ValueError(
            f"Team '{team_name}' was not found. "
            f"Available teams: {available_teams}"
        )

    if len(team_rows) != 1:
        raise ValueError(
            f"Expected one vector for '{team_name}', "
            f"but found {len(team_rows)}."
        )

    return (
        team_rows
        .iloc[0]
        .loc[statistical_columns]
        .copy()
    )

In [46]:
def build_current_match_vector(
    home_team,
    away_team,
    team_vectors,
    statistical_columns,
    team_id_column="country_clean",
):
    """
    Build a base current match vector from two national teams.

    Parameters
    ----------
    home_team : str
        Name of the home national team.

    away_team : str
        Name of the away national team.

    team_vectors : pandas.DataFrame
        Dataset containing one aggregated vector per team.

    statistical_columns : list of str
        Statistical feature columns used to represent each team.

    team_id_column : str, default="country_clean"
        Column used to identify each team.

    Returns
    -------
    pandas.DataFrame
        Single-row match vector containing home and away features.

    Raises
    ------
    ValueError
        If the same team is used as both home and away.
    """

    if home_team == away_team:
        raise ValueError(
            "Home team and away team must be different."
        )

    home_vector = get_team_vector(
    team_name=home_team,
    team_vectors=team_vectors,
    statistical_columns=statistical_columns,
    team_id_column=team_id_column,
    )

    away_vector = get_team_vector(
    team_name=away_team,
    team_vectors=team_vectors,
    statistical_columns=statistical_columns,
    team_id_column=team_id_column,
    )
    
    home_vector.index = [
        f"home_{column}"
        for column in home_vector.index
    ]

    away_vector.index = [
        f"away_{column}"
        for column in away_vector.index
    ]

    match_vector = pd.concat(
        [
            pd.Series(
                {
                    "home_team": home_team,
                    "away_team": away_team,
                }
            ),
            home_vector,
            away_vector,
        ]
    )

    return match_vector.to_frame().T

In [32]:
def get_team_statistics(df):
    """
    Return the numeric statistics available for both
    home and away teams.

    Identifiers, team names, target variables, and
    match results are excluded.
    """

    excluded_stats = {
        "match_id",
        "team_name",
        "team",
        "score",
        "year",
    }

    home_stats = {
        column.replace("home_", "", 1)
        for column in df.columns
        if column.startswith("home_")
    }

    away_stats = {
        column.replace("away_", "", 1)
        for column in df.columns
        if column.startswith("away_")
    }

    common_stats = sorted(
        home_stats.intersection(away_stats)
    )

    valid_stats = [
        stat
        for stat in common_stats
        if stat not in excluded_stats
        and pd.api.types.is_numeric_dtype(
            df[f"home_{stat}"]
        )
        and pd.api.types.is_numeric_dtype(
            df[f"away_{stat}"]
        )
    ]

    return valid_stats

In [33]:
def create_difference_features(df):
    """
    Create absolute differences between home and away
    team statistics.

    Formula
    -------
    diff_X = home_X - away_X
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        df_c[f"diff_{stat}"] = (
            df_c[f"home_{stat}"]
            - df_c[f"away_{stat}"]
        )

    return df_c

In [34]:
def create_relative_difference_features(
    df,
    epsilon=1e-6,
):
    """
    Create relative differences between home and away
    team statistics.

    Formula
    -------
    relative_diff_X =
        (home_X - away_X)
        / (home_X + away_X + epsilon)
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        home_col = f"home_{stat}"
        away_col = f"away_{stat}"
        new_col = f"relative_diff_{stat}"

        df_c[new_col] = (
            df_c[home_col] - df_c[away_col]
        ) / (
            df_c[home_col]
            + df_c[away_col]
            + epsilon
        )

    return df_c

In [35]:
def create_sum_features(df):
    """
    Create combined sums of home and away
    team statistics.

    Formula
    -------
    sum_X = home_X + away_X
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        home_col = f"home_{stat}"
        away_col = f"away_{stat}"
        new_col = f"sum_{stat}"

        df_c[new_col] = (
            df_c[home_col] + df_c[away_col]
        )

    return df_c

### Master Function

In [44]:
def build_prediction_ready_match_vector(
    home_team,
    away_team,
    team_vectors,
    statistical_columns,
    historical_feature_columns,
    team_id_column="country_clean",
):
    base_vector = build_current_match_vector(
        home_team=home_team,
        away_team=away_team,
        team_vectors=team_vectors,
        statistical_columns=statistical_columns,
        team_id_column=team_id_column,
    )

    home_away_columns = [
        column
        for column in base_vector.columns
        if column.startswith(("home_", "away_"))
        and column not in {"home_team", "away_team"}
    ]

    base_vector[home_away_columns] = (
        base_vector[home_away_columns]
        .astype(float)
    )

    engineered_vector = (
        base_vector
        .pipe(create_difference_features)
        .pipe(create_relative_difference_features)
        .pipe(create_sum_features)
    )

    missing_features = [
        column
        for column in historical_feature_columns
        if column not in engineered_vector.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing required historical features: "
            f"{missing_features}"
        )

    return engineered_vector[
        ["home_team", "away_team"]
        + historical_feature_columns
    ].copy()

## Load Model Artifacts

In [5]:
rf_artifact = joblib.load(RF_MODEL_PATH)
xgb_artifact = joblib.load(XGB_MODEL_PATH)

In [6]:
rf_model = rf_artifact["model"]
rf_feature_names = rf_artifact["feature_names"]
rf_target_classes = rf_artifact["target_classes"]
rf_model_version = rf_artifact["model_version"]

xgb_model = xgb_artifact["model"]
xgb_feature_names = xgb_artifact["feature_names"]
xgb_target_classes = xgb_artifact["target_classes"]
xgb_model_version = xgb_artifact["model_version"]

In [10]:
rf_only = sorted(set(rf_feature_names) - set(xgb_feature_names))
xgb_only = sorted(set(xgb_feature_names) - set(rf_feature_names))

print(f"RF features: {len(rf_feature_names)}")
print(f"XGB features: {len(xgb_feature_names)}")

print("\nOnly in Random Forest:")
print(rf_only)

print("\nOnly in XGBoost:")
print(xgb_only)

RF features: 124
XGB features: 124

Only in Random Forest:
['relative_diff_Ball Receipt*']

Only in XGBoost:
['home_Player On']


In [12]:
print("RF target classes:", rf_target_classes)
print("XGB target classes:", xgb_target_classes)

RF target classes: [np.int64(-1), np.int64(0), np.int64(1)]
XGB target classes: [np.int64(0), np.int64(1), np.int64(2)]


In [13]:
assert hasattr(rf_model, "predict")
assert hasattr(rf_model, "predict_proba")

assert hasattr(xgb_model, "predict")
assert hasattr(xgb_model, "predict_proba")

print(f"Random Forest: {rf_model_version}")
print(f"Random Forest features: {len(rf_feature_names)}")
print(f"Random Forest target classes: {rf_target_classes}")

print(f"\nXGBoost: {xgb_model_version}")
print(f"XGBoost features: {len(xgb_feature_names)}")
print(f"XGBoost target classes: {xgb_target_classes}")

print("\nBoth model artifacts loaded successfully.")

Random Forest: random_forest_v03
Random Forest features: 124
Random Forest target classes: [np.int64(-1), np.int64(0), np.int64(1)]

XGBoost: xgboost_v04
XGBoost features: 124
XGBoost target classes: [np.int64(0), np.int64(1), np.int64(2)]

Both model artifacts loaded successfully.


## Class Label Mapping

In [14]:
RF_CLASS_LABELS = {
    -1: "Away Win",
    0: "Draw",
    1: "Home Win",
}

In [15]:
XGB_CLASS_LABELS = {
    0: "Away Win",
    1: "Draw",
    2: "Home Win",
}

In [16]:
assert set(rf_target_classes) == set(RF_CLASS_LABELS.keys())
assert set(xgb_target_classes) == set(XGB_CLASS_LABELS.keys())

print("Class label mappings validated successfully.")

Class label mappings validated successfully.


## Load Current Team Vectors

In [26]:
current_team_vectors = pd.read_csv(TEAM_VECTORS_PATH)

current_team_vectors = current_team_vectors.rename(
    columns={"country_clean": "team_name"}
)

### Validate Current Team Vectors

In [27]:
assert not current_team_vectors.empty
assert "team_name" in current_team_vectors.columns
assert current_team_vectors["team_name"].is_unique
assert not current_team_vectors.isna().any().any()

print(f"Current team vectors shape: {current_team_vectors.shape}")
print(f"Available teams: {current_team_vectors['team_name'].nunique()}")
print("Current team vectors loaded successfully.")

Current team vectors shape: (48, 32)
Available teams: 48
Current team vectors loaded successfully.


### Inspect Available Teams

In [28]:
available_teams = sorted(
    current_team_vectors["team_name"].unique()
)

display(
    pd.DataFrame({
        "team_name": available_teams
    })
)

,team_name
0,Algeria
1,Argentina
2,Australia
3,Austria
4,Belgium
5,Bosnia And Herzegovina
6,Brazil
7,Cabo Verde
8,Canada
9,Colombia


In [29]:
assert not current_team_vectors.empty
assert "team_name" in current_team_vectors.columns
assert current_team_vectors["team_name"].is_unique

# Team Selection

Select the home and away teams for prediction.

In [37]:
home_team = "Argentina"
away_team = "Brazil"

In [38]:
assert home_team in current_team_vectors["team_name"].values
assert away_team in current_team_vectors["team_name"].values
assert home_team != away_team

print(f"Home team: {home_team}")
print(f"Away team: {away_team}")
print("Team selection validated successfully.")

Home team: Argentina
Away team: Brazil
Team selection validated successfully.


In [39]:
selected_team_vectors = current_team_vectors[
    current_team_vectors["team_name"].isin(
        [home_team, away_team]
    )
]

display(selected_team_vectors)

,team_name,squad_size,matched_players,unmatched_players,coverage,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,...,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
1,Argentina,26,20,6,0.769231,91.0,28.0,26859.0,1799.0,591.0,...,18.0,18.0,4893.0,19.0,1022.0,11.0,5.0,2.0,0.0,0.0
6,Brazil,26,18,8,0.692308,30.0,9.0,8966.0,704.0,240.0,...,12.0,12.0,1902.0,10.0,235.0,4.0,5.0,0.0,0.0,0.0


# Prediction Pipeline

### Statistical Columns Configuration

In [41]:
TEAM_METADATA_COLUMNS = [
    "team_name",
    "squad_size",
    "matched_players",
    "unmatched_players",
    "coverage",
]

In [42]:
statistical_columns = [
    column
    for column in current_team_vectors.columns
    if column not in TEAM_METADATA_COLUMNS
]

print(f"Statistical columns: {len(statistical_columns)}")
print()
print(statistical_columns)

Statistical columns: 27

['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Error', 'Offside', 'Own Goal Against', 'Camera On', 'Own Goal For']


## Build Random Forest Prediction Vector

In [48]:
rf_prediction_vector = build_prediction_ready_match_vector(
    home_team=home_team,
    away_team=away_team,
    team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    historical_feature_columns=rf_feature_names,
    team_id_column="team_name",
)

### Validate Random Forest Prediction Vector

In [49]:
assert rf_prediction_vector.shape == (1, 126)

assert list(
    rf_prediction_vector.columns[2:]
) == rf_feature_names

assert not rf_prediction_vector.isna().any().any()

print("Random Forest prediction vector validated successfully.")

Random Forest prediction vector validated successfully.


## Build XGBoost Prediction Vector

In [50]:
xgb_prediction_vector = build_prediction_ready_match_vector(
    home_team=home_team,
    away_team=away_team,
    team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    historical_feature_columns=xgb_feature_names,
    team_id_column="team_name",
)

### Validate XGBoost Prediction Vector

In [51]:
assert xgb_prediction_vector.shape == (1, 126)

assert list(
    xgb_prediction_vector.columns[2:]
) == xgb_feature_names

assert not xgb_prediction_vector.isna().any().any()

print("XGBoost prediction vector validated successfully.")

XGBoost prediction vector validated successfully.


# Model Inference

## Random Forest Prediction

In [52]:
rf_X = rf_prediction_vector[rf_feature_names]

rf_predicted_class = rf_model.predict(rf_X)[0]
rf_probabilities = rf_model.predict_proba(rf_X)[0]

rf_decoded_prediction = RF_CLASS_LABELS[rf_predicted_class]

### Random Forest Probabilities

In [53]:
rf_probability_table = pd.DataFrame({
    "class": rf_target_classes,
    "result": [
        RF_CLASS_LABELS[class_value]
        for class_value in rf_target_classes
    ],
    "probability": rf_probabilities,
})

rf_probability_table

,class,result,probability
0,-1,Away Win,0.360
1,0,Draw,0.355
2,1,Home Win,0.285


## XGBoost Prediction

In [54]:
xgb_X = xgb_prediction_vector[xgb_feature_names]

xgb_predicted_class = xgb_model.predict(xgb_X)[0]
xgb_probabilities = xgb_model.predict_proba(xgb_X)[0]

xgb_decoded_prediction = XGB_CLASS_LABELS[xgb_predicted_class]

### XGBoost Probabilities

In [55]:
xgb_probability_table = pd.DataFrame({
    "class": xgb_target_classes,
    "result": [
        XGB_CLASS_LABELS[class_value]
        for class_value in xgb_target_classes
    ],
    "probability": xgb_probabilities,
})

xgb_probability_table

,class,result,probability
0,0,Away Win,0.493258
1,1,Draw,0.223666
2,2,Home Win,0.283077


## Validate Inference Outputs

In [56]:
assert len(rf_probabilities) == len(rf_target_classes)
assert len(xgb_probabilities) == len(xgb_target_classes)

assert np.isclose(rf_probabilities.sum(), 1.0)
assert np.isclose(xgb_probabilities.sum(), 1.0)

assert rf_predicted_class in rf_target_classes
assert xgb_predicted_class in xgb_target_classes

print("Model inference validated successfully.")

Model inference validated successfully.


## Model Agreement

In [57]:
models_agree = (
    rf_decoded_prediction == xgb_decoded_prediction
)

print(f"Random Forest prediction: {rf_decoded_prediction}")
print(f"XGBoost prediction: {xgb_decoded_prediction}")
print(f"Models agree: {models_agree}")

Random Forest prediction: Away Win
XGBoost prediction: Away Win
Models agree: True


# Prediction Report

## Match Summary

In [58]:
print("=" * 60)
print("WORLD CUP 2026 MATCH PREDICTION")
print("=" * 60)
print()

print(f"Home Team : {home_team}")
print(f"Away Team : {away_team}")
print()

WORLD CUP 2026 MATCH PREDICTION

Home Team : Argentina
Away Team : Brazil



## Random Forest Report

In [59]:
print("Random Forest")
print("-" * 60)

display(
    rf_probability_table.sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)
)

print(f"Prediction: {rf_decoded_prediction}")
print(
    f"Confidence: {rf_probabilities.max():.2%}"
)

Random Forest
------------------------------------------------------------


,class,result,probability
0,-1,Away Win,0.360
1,0,Draw,0.355
2,1,Home Win,0.285


Prediction: Away Win
Confidence: 36.00%


## XGBoost Report

In [60]:
print("XGBoost")
print("-" * 60)

display(
    xgb_probability_table.sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)
)

print(f"Prediction: {xgb_decoded_prediction}")
print(
    f"Confidence: {xgb_probabilities.max():.2%}"
)

XGBoost
------------------------------------------------------------


,class,result,probability
0,0,Away Win,0.493258
1,2,Home Win,0.283077
2,1,Draw,0.223666


Prediction: Away Win
Confidence: 49.33%


## Final Prediction Summary

In [61]:
summary_df = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
    ],
    "Prediction": [
        rf_decoded_prediction,
        xgb_decoded_prediction,
    ],
    "Confidence": [
        rf_probabilities.max(),
        xgb_probabilities.max(),
    ],
})

summary_df["Confidence"] = (
    summary_df["Confidence"] * 100
).round(2)

display(summary_df)

print()

if models_agree:
    print("Consensus: Both models predict the same outcome.")
else:
    print("Consensus: Models disagree.")

,Model,Prediction,Confidence
0,Random Forest,Away Win,36.00
1,XGBoost,Away Win,49.33



Consensus: Both models predict the same outcome.


# Discussion

### English

- The trained Random Forest and XGBoost models were successfully integrated into a reusable inference pipeline.

- Current team vectors were transformed into prediction-ready match vectors using the same feature engineering process employed during model training, ensuring complete compatibility between historical and inference data.

- Both models produced valid probability distributions and generated consistent predictions for the selected match. Although the estimated probabilities differed, both models predicted the same outcome, demonstrating that the complete inference workflow operates correctly.

- The notebook validates the project's deployment architecture by separating model training from model inference through reusable serialized artifacts.


### Español

- Los modelos entrenados Random Forest y XGBoost fueron integrados exitosamente dentro de un pipeline reutilizable de inferencia.

- Los vectores actuales de los equipos fueron transformados en vectores de partido listos para predicción utilizando exactamente el mismo proceso de ingeniería de variables empleado durante el entrenamiento, garantizando la compatibilidad entre los datos históricos y los datos de inferencia.

- Ambos modelos produjeron distribuciones de probabilidad válidas y generaron predicciones consistentes para el partido seleccionado. Aunque las probabilidades estimadas fueron diferentes, ambos modelos coincidieron en el resultado predicho, demostrando que el flujo completo de inferencia funciona correctamente.

- Esta notebook valida la arquitectura de despliegue del proyecto al separar el entrenamiento de modelos del proceso de inferencia mediante artefactos reutilizables.

# Conclusion

### English

- The first end-to-end inference pipeline was successfully implemented.

- Previously trained machine learning models can now be reused without retraining.

- Current team vectors are automatically transformed into prediction-ready feature vectors.

- Both Random Forest and XGBoost successfully generated predictions and probability estimates.

- The complete inference workflow was validated from team selection to final prediction reporting.

- The project is now prepared for the next stage, focused on deployment, API stabilization, and additional model improvements.


### Español

- Se implementó exitosamente el primer pipeline completo de inferencia.

- Los modelos entrenados pueden reutilizarse sin necesidad de volver a entrenarlos.

- Los vectores actuales de los equipos se transforman automáticamente en vectores listos para predicción.

- Tanto Random Forest como XGBoost generaron correctamente predicciones y distribuciones de probabilidad.

- Se validó el flujo completo de inferencia, desde la selección de equipos hasta el reporte final de resultados.

- El proyecto queda preparado para la siguiente etapa, enfocada en la estabilización de la API, el despliegue y futuras mejoras de los modelos.
